In [4]:
# Import necessary libraries and modules
import os
import glob
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder

In [5]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

print("API key loaded")

API key loaded


#### Documents collections

In [7]:
documents = []

for pdf_path in glob.glob("documents/*.pdf"):  # adjust folder path
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    documents.extend(docs)

print(f"Loaded {len(documents)} PDF Documents.")


Loaded 5 PDF Documents.


#### Text Splitters

In [8]:
# Create splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=20,
    length_function=len
)

# Split documents
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}: {chunk.page_content}")

Split 5 documents into 18 chunks

Chunk 1: Project  1:  Car  Price  Prediction  System  Developed  a  supervised  machine  learning  model  to  predict  car  prices  using  historical  sales  data.  The  project  involved  data  cleaning,  feature  engineering,  model  training,  and  evaluation.  Deployed  the  model  using  Flask.

Chunk 2: using  Flask.   Project  2:  Retail  Sales  Forecasting  Built  a  time-series  forecasting  model  to  predict  monthly  sales  trends  for  an  e-commerce  platform.  Used  historical  transaction  data  and  implemented  ARIMA  and  machine  learning-based  approaches.   Project  3:

Chunk 3: Project  3:  Retrieval-Augmented  Generation  (RAG)  Chatbot  Designed  a  document-based  question  answering  system  using  LangChain,  vector  embeddings,  and  a  large  language  model.  The  chatbot  retrieves  relevant  document  chunks  before  generating  accurate  responses.   Project

Chunk 4: Project  4:  Student  Performance  Prediction  Crea

#### Embeddings

In [9]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=api_key
)

# Test embedding
test_embedding = embeddings.embed_query("What is RAG?")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

Embedding dimension: 1536
First 5 values: [0.0006281227106228471, 0.02569717727601528, 0.007161187008023262, 0.03336399793624878, -0.031968604773283005]


#### Vector Store
# Create vector store from documents

In [11]:
vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    collection_name="my_info_collection",
    persist_directory="./chroma_db"
)

In [12]:
# Test retriver
query = "Technical skills"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke(query)
results


[Document(metadata={'producer': 'Skia/PDF m145 Google Docs Renderer', 'page_label': '1', 'source': 'documents\\Professional Resume.pdf', 'page': 0, 'creator': 'PyPDF', 'total_pages': 1, 'creationdate': '', 'title': 'Professional Resume'}, page_content='M.Sc.  Artificial  Intelligence  -  B.Sc.  Applied  Mathematics   Technical  Skills:  -  Programming:  Python,  SQL,  JavaScript  -  Machine  Learning:  Regression,  Classification,  Time-Series  Forecasting  -  Deep  Learning:  Neural  Networks,  Transformers  -  NLP:  Text  Embeddings,  Vector'),
 Document(metadata={'page': 0, 'creator': 'PyPDF', 'page_label': '1', 'title': 'Professional Resume', 'source': 'documents\\Professional Resume.pdf', 'total_pages': 1, 'producer': 'Skia/PDF m145 Google Docs Renderer', 'creationdate': ''}, page_content='M.Sc.  Artificial  Intelligence  -  B.Sc.  Applied  Mathematics   Technical  Skills:  -  Programming:  Python,  SQL,  JavaScript  -  Machine  Learning:  Regression,  Classification,  Time-Series

#### Conversational Rag

In [13]:
# Create LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=api_key
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

# prompt
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant answering questions about Alex Morgan using the provided documents.

Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't know."

<context>
{context}
</context>

Question: {question}

Answer in clear sentences.
At the end, list the sources you used as bullet points.
""")

# format documents
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in docs
    )

# RAG chain Using LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [14]:
query = "What AI projects has Alex worked on?"
response = rag_chain.invoke(query)
print(response)

Alex Morgan has worked on designing, training, and deploying machine learning models as an AI Engineer. Some of the projects Alex has worked on include using Python, SQL, LangChain, PyTorch, and cloud-based ML pipelines.

Sources:
- documents\Professional Resume.pdf


#### Conversational RAG

In [16]:
# Store for chat histories
chat_store = {}

def get_session_history(session_id: str):
    if session_id not in chat_store:
        chat_store[session_id] = InMemoryChatMessageHistory()
    return chat_store[session_id]

# Create conversational prompt
conv_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant answering questions about Alex Morgan using the provided documents. Use ONLY the context below to answer the question. If the answer is not in the context, say I don't know."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("system", "Answer in clear sentences. At the end, list the sources you used as bullet points."),
    ("human", "Context: {context}\n\nQuestion: {question}")
])

# Build base chain
conv_chain_base = (
    RunnableParallel(
        context=lambda x: format_docs(retriever.invoke(x["question"])),
        question=lambda x: x["question"],
        chat_history=lambda x: x.get("chat_history", [])
    )
    | conv_prompt
    | llm
    | StrOutputParser()
)

# Wrap with message history
conv_chain = RunnableWithMessageHistory(
    conv_chain_base,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)



**Questions**

In [18]:
# First question
response = conv_chain.invoke(
    {"question": "What projects has Alex worked on?"},
    config={"configurable": {"session_id": "user_1"}}
)
print("Response 1:\n", response)

# Follow-up question
response2 = conv_chain.invoke(
    {"question": "Which of those involve RAG systems?"},
    config={"configurable": {"session_id": "user_1"}}
)

print("\nResponse 2:\n", response2)

Response 1:
 Alex Morgan has worked on designing, training, and deploying machine learning models.

Sources:
- Professional Resume.pdf

Response 2:
 Alex Morgan's work involving RAG systems includes building predictive models for sales forecasting in e-commerce, developing AI-powered chatbots for document-based question answering, and designing REST APIs for ML models.

Sources:
- Professional Resume.pdf
